In [11]:
import json
import os 

with open('config.json', 'r') as f:
    data = json.load(f)

pathway_gen = os.path.abspath(data["python_files"])
pathway_temp = os.path.abspath(data["publications"])
pathway = os.path.join(pathway_temp, "broihanne2019monkeys")
original_data_pathway = os.path.join(pathway, "original_data")

complete_path_1 = os.path.join(original_data_pathway, "Broihanne_2019_data.csv")

out_pathway = os.path.join(pathway, "standardized_data")
if not os.path.exists(out_pathway):
    os.makedirs(out_pathway)

In [12]:
import pandas as pd
import numpy as np
import pyreadstat

df = pd.read_csv(complete_path_1)

df['study_id']="broihanne2019monkeys"
df.columns = map(str.lower, df.columns)
df=df.applymap(lambda s: s.lower() if type(s) == str else s)
df.rename(columns={ "subject":"participant"}, inplace=True)

In [13]:
comp_path_name_errors = os.path.join(pathway_gen, "broihanne2019monkeys_includeindatabase.csv")

df_name  = pd.read_csv(comp_path_name_errors)
for x,y in zip(df_name['Subject number'],df_name['name']):
    df['participant'].replace(x, y, inplace=True)
for x,y in zip(df_name['Species number'],df_name['species']):
    df['species'].replace(x, y, inplace=True)
df=df.applymap(lambda s: s.lower() if type(s) == str else s)

comp_path_ape_info = os.path.join(pathway_gen, "apes_includeindatabase.csv")
apedf = pd.read_csv(comp_path_ape_info)    
df= df.merge(apedf,left_on='participant', right_on='name', how='left')

df.rename(columns={"exchange (1 yes, 0 no)":"exchange_1-yes_0-no",
                   "sex":"sex_temp"}, inplace=True)

In [14]:
# df.columns
# df['participant'].unique()

In [15]:
complete_path_age = os.path.join(original_data_pathway, "subject_list.csv")
subject_list = pd.read_csv(complete_path_age)   
df= df.merge(subject_list,left_on='participant', right_on='name', how='left')
df.rename(columns={"age": "age_in_years",
                   "expected value":'expected_value'}, inplace=True)

df['species'].replace('gorilla   ', 'gorilla', inplace=True, regex=True)
df['species'].unique()

array(['tonkean_macaque', 'brown_capuchin', 'bonobo', 'gorilla',
       'orangutan', 'chimpanzee'], dtype=object)

In [16]:
studyID_standardized=df[['study_id',  'participant','age_in_years', 'sex', 
                         'species','session', 'lotterie',
       'exchange_1-yes_0-no', 'expected_value', 'outcomepreceedingtype',
       'outcomecumulatedtype']]
comp_out_path_stand = os.path.join(out_pathway, 'broihanne2019monkeys_standardized.csv')
studyID_standardized.to_csv(comp_out_path_stand, encoding='utf-8-sig', index=False)


names =studyID_standardized.columns.tolist()
df = pd.DataFrame(names)
df = df.rename(columns={0: "column_name"})
df["description"] = ""
studyID_glossary=df[["column_name", "description"]]

comp_out_path_glossary = os.path.join(out_pathway, 'broihanne2019monkeys_glossary.csv')
studyID_glossary.to_csv(comp_out_path_glossary, encoding='utf-8-sig', index=False)